### Модели ходьбы. Capture step


#### Linear Inverted Pendulum

Реализуйте контроллер, переставляющий ногу раз в полсекунды так, чтобы LIP ходил за трекбаром (см. видео в чате).

In [73]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


class LIP:
    def __init__(self, m=1.0, l=1.0, g=10.0, x_contact0=0.2, x0=0.3, xd0=-0.1, dt=0.01, WIND_X=700, scale=100):
        self.m = m
        self.l = l
        self.g = g

        self.x = np.array([[x0], [xd0]])
        self.dt = dt

        self.WIND_X = WIND_X
        self.scale = scale

        self.x_contact = x_contact0

    def change_x_contact(self, new_x_contact):
        self.x[0, 0] += self.x_contact - new_x_contact

        self.x_contact = new_x_contact

    def get_state(self):
        return self.x

    def propagate_system(self):
        g = self.g
        l = self.l

        xdd = g / self.l * self.x[0, 0]

        self.x[1, 0] += xdd * self.dt
        self.x[0, 0] += self.x[1, 0] * self.dt

    def draw(self, cx, cy, scale=100, color=(234, 123, 123), canvas=None):
        if canvas is None:
            canvas = np.ones((700, 700, 3)) * 0

        wind_h, wind_w, _ = canvas.shape

        cv2.line(canvas, (0, wind_h // 2), (wind_w, wind_h // 2), color, 2)

        cv2.circle(
            canvas,
            (int(wind_w // 2 + (self.x_contact + self.x[0, 0]) * scale), int(wind_h // 2 - self.l * scale)),
            7,
            color,
            2,
        )

        cv2.line(
            canvas,
            (int(wind_w // 2 + (self.x_contact + self.x[0, 0]) * scale), int(wind_h // 2 - self.l * scale)),
            (int(wind_w // 2 + self.x_contact * scale), int(wind_h // 2)),
            color,
            2,
        )

        return canvas

# https://arxiv.org/pdf/2208.01786
def get_lip_step(state: np.ndarray, g: float, l: float, T: float, target_vel: float):
    lam = np.sqrt(g / l)

    sigma1 = lam / np.tanh(lam * T / 2)
    M = np.array(
        [[np.cosh(lam * T), np.sinh(lam * T) / lam], [lam * np.sinh(lam * T), np.cosh(lam * T)]],
        dtype=np.float32,
    )
    a = np.eye(2, dtype=np.float32)
    b = np.array([-1, 0], dtype=np.float32)
    A = M @ a
    B = M @ b[:, None]

    a11, a12, a21, a22 = A[0, 0], A[0, 1], A[1, 0], A[1, 1]
    b1, b2 = B[0, 0], B[1, 0]
    system = np.array(
        [
            [b1, b2],
            [a22 * b1 - a12 * b2, a11 * b2 - a21 * b1],
        ],
        dtype=np.float32,
    )
    rhs = np.array([-a11 - a22, -a11 * a22 + a12 * a21])

    K = np.linalg.solve(system, rhs)

    x_star = np.array([1, sigma1], dtype=np.float32) * target_vel * T / 2
    u_star = target_vel * T

    e = state.squeeze(1) - x_star
    return u_star + K @ e


def run_LIP_episode(x0=1.2, xd0=-0.1, scale=30, dt=0.05, WIND_X=700, WIND_Y=700):
    canvas = np.zeros((WIND_Y, WIND_X, 3), np.uint8)

    lip = LIP(m=1.0, l=1.0, g=10.0, x_contact0=0, x0=x0, xd0=xd0, dt=dt, WIND_X=WIND_X, scale=scale)

    iter_num = 75000
    i = 1

    x_traj = []
    u_traj = []

    cv2.namedWindow("LIP")
    cv2.createTrackbar("target", "LIP", WIND_X // 2, WIND_X, lambda i: i)
    period = 10

    while True:
        state = lip.get_state()

        x_traj.append(state[0, 0])

        if i % period == 0:
            target_pos = (cv2.getTrackbarPos("target", "LIP") - WIND_X // 2) / scale

            # print("contact", lip.x_contact)
            # print("state[0, 0]", state[0, 0])
            # print("target", target_pos)

            # YOUR CODE BELOW
            T = dt * period
            pos_world = lip.x_contact + state[0, 0]
            target_vel = np.clip(target_pos - pos_world, -2, 2)
            step = get_lip_step(state, lip.g, lip.l, T, target_vel)
            control = lip.x_contact + step
            # YOUR CODE ABOVE

            lip.change_x_contact(control)
            u_traj.append(control)
        else:
            u_traj.append(0)

        lip.propagate_system()

        canvas = cv2.addWeighted(canvas, 0.93, canvas, 0, 0)

        lip.draw(WIND_X // 2, WIND_Y // 2, scale=scale, canvas=canvas)

        cv2.imshow("LIP", canvas)

        i += 1

        if i > iter_num:
            break

        key = cv2.waitKey(int(1000 * dt)) & 0xFF

        if key == ord("q"):
            break

    cv2.destroyAllWindows()
    cv2.waitKey(10)

    return x_traj, u_traj


x_hist, u_hist = run_LIP_episode(x0=0.4, xd0=0.1, scale=200, WIND_X=1300, WIND_Y=600)

#### Capture step

Реализуйте capture step (ловящий шаг) для LIP модели.
На вход алгоритму поступает состояние LIP.
Нужно найти такую точку постановки ноги, что через полсекунды скорость маятника будет минимальна. Это можно сделать с помощью перебора по решетке (grid search).

In [74]:
from functools import partial

lip = LIP(m=1.0, l=1.0, g=10.0, x_contact0=0, x0=0.4, xd0=0.1, dt=0.05, WIND_X=1300, scale=200)
capture_step = partial(get_lip_step, g=lip.g, l=lip.l, T=0.5, target_vel=0)

state = lip.get_state()

step = capture_step(state)
print(step)

0.43441824018955233
